# Notebook 2: CoinGecko Crypto Analysis

**Alpha Search — Real Data Only**

This notebook fetches real cryptocurrency price data from the **CoinGecko** free API and performs technical analysis including returns, volatility, momentum, and a simple mean-reversion backtest.

**Data Sources:**
- **CoinGecko API** (api.coingecko.com) — Free tier: 10-30 calls/minute

**Assets Analyzed:**
| Asset | CoinGecko ID | Description |
|-------|-------------|-------------|
| Bitcoin | `bitcoin` | BTC - Digital gold |
| Ethereum | `ethereum` | ETH - Smart contract platform |
| Solana | `solana` | SOL - High-performance L1 |

**Analysis Performed:**
1. Price normalization and comparison
2. Daily returns and rolling volatility
3. Correlation matrix
4. Momentum signals (20-day Rate of Change)
5. Mean-reversion strategy (z-score based)
6. Simple backtest: buy z-score < -2, sell z-score > +2

**Rate Limits:** CoinGecko free tier allows 10-30 calls/min. We use `time.sleep(1.5)` between calls.

In [ ]:
# Install Alpha Search
!pip install git+https://github.com/alpha-search/alpha-search.git -q

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

In [ ]:
# CoinGecko Data Fetcher Function
# Uses the free CoinGecko API - no API key needed

def fetch_crypto(coin_id='bitcoin', days=365, label=None, _retry_count=0):
    """
    Fetch cryptocurrency price data from CoinGecko (free, no API key).
    
    Parameters:
        coin_id: CoinGecko coin identifier (e.g., 'bitcoin', 'ethereum')
        days: Number of days of history to fetch
        label: Human-readable name
    
    Returns:
        pd.Series with datetime index and USD prices
    """
    url = f'https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart'
    params = {'vs_currency': 'usd', 'days': days}
    resp = None
    try:
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        
        if 'prices' not in data:
            print(f"  ERROR: No price data for {coin_id}. Response: {data}")
            return pd.Series(name=label or coin_id)
        
        prices = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
        prices['timestamp'] = pd.to_datetime(prices['timestamp'], unit='ms')
        series = prices.set_index('timestamp')['price']
        series.name = label or coin_id
        tag = label or coin_id
        print(f"  Fetched {tag}: {len(series)} daily prices, latest ${series.iloc[-1]:,.2f}")
        return series
    except requests.exceptions.HTTPError as e:
        if resp is not None and resp.status_code == 429:
            if _retry_count < 3:
                print(f"  RATE LIMITED for {coin_id}. Waiting 60s then retrying...")
                time.sleep(60)
                return fetch_crypto(coin_id, days, label, _retry_count=_retry_count + 1)
            else:
                print(f"  RATE LIMITED for {coin_id}. Max retries exceeded.")
        else:
            print(f"  HTTP ERROR fetching {coin_id}: {e}")
        return pd.Series(name=label or coin_id)
    except Exception as e:
        print(f"  ERROR fetching {coin_id}: {e}")
        return pd.Series(name=label or coin_id)

print('CoinGecko fetcher function defined.')

In [ ]:
# Fetch BTC, ETH, SOL prices (last 365 days)
print('Fetching cryptocurrency prices from CoinGecko...')
print('=' * 60)

crypto_data = {}

crypto_data['BTC'] = fetch_crypto('bitcoin', days=365, label='Bitcoin')
time.sleep(1.5)  # Respect rate limit

crypto_data['ETH'] = fetch_crypto('ethereum', days=365, label='Ethereum')
time.sleep(1.5)

crypto_data['SOL'] = fetch_crypto('solana', days=365, label='Solana')

print('=' * 60)
fetched = [k for k, v in crypto_data.items() if len(v) > 0]
print(f'Fetched {len(fetched)}/3 coins successfully: {fetched}')

In [ ]:
# Normalize all prices to start at 100 for comparison
fig, ax = plt.subplots(figsize=(12, 7))

colors = {'BTC': '#F7931A', 'ETH': '#627EEA', 'SOL': '#14F195'}

for ticker, prices in crypto_data.items():
    if len(prices) == 0:
        continue
    normalized = (prices / prices.iloc[0]) * 100
    ax.plot(normalized.index, normalized.values, 
            label=f'{ticker} (${prices.iloc[-1]:,.0f})', 
            color=colors.get(ticker, 'gray'), linewidth=2)

ax.axhline(y=100, color='black', linestyle='--', alpha=0.5, label='Baseline (100)')
ax.set_title('Crypto Performance (Normalized to 100)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Normalized Price (Start = 100)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Add performance annotations
for ticker, prices in crypto_data.items():
    if len(prices) == 0:
        continue
    ret = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
    ax.annotate(f'{ret:+.1f}%', 
                xy=(prices.index[-1], (prices.iloc[-1]/prices.iloc[0])*100),
                fontsize=10, fontweight='bold',
                color='green' if ret > 0 else 'red')

plt.tight_layout()
plt.show()
print('All prices normalized to 100 at the start of the period.')

In [ ]:
# Calculate Daily Returns and Volatility Statistics
returns_data = {}
vol_stats = []

print('DAILY RETURNS ANALYSIS')
print('=' * 60)

for ticker, prices in crypto_data.items():
    if len(prices) == 0:
        continue
    # Resample to daily and calculate returns
    daily = prices.resample('D').last().dropna()
    ret = daily.pct_change().dropna() * 100
    returns_data[ticker] = ret
    
    stats = {
        'Asset': ticker,
        'Mean Daily Return': ret.mean(),
        'Daily Volatility': ret.std(),
        'Annualized Vol': ret.std() * np.sqrt(365),
        'Best Day': ret.max(),
        'Worst Day': ret.min(),
        'Sharpe (raw)': ret.mean() / ret.std() if ret.std() > 0 else 0,
    }
    vol_stats.append(stats)

vol_df = pd.DataFrame(vol_stats)
display(vol_df.round(2))
print()
print('All calculations on REAL CoinGecko price data.')

In [ ]:
# Plot Rolling 30-Day Volatility
fig, ax = plt.subplots(figsize=(12, 6))

for ticker, ret in returns_data.items():
    rolling_vol = ret.rolling(30).std() * np.sqrt(365)
    ax.plot(rolling_vol.index, rolling_vol.values, 
            label=f'{ticker}', linewidth=1.5, color=colors.get(ticker, 'gray'))

ax.set_title('Rolling 30-Day Annualized Volatility', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Annualized Volatility (%)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Volatility spikes during market stress events.')

In [ ]:
# Correlation Matrix Heatmap
returns_df = pd.DataFrame(returns_data)
corr = returns_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=1, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax.set_title('Crypto Daily Returns Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Correlation based on {len(returns_df)} days of real CoinGecko data.')

In [ ]:
# Momentum Signals: 20-Day Rate of Change
fig, axes = plt.subplots(len(crypto_data), 1, figsize=(12, 3 * len(crypto_data)), sharex=True)
if len(crypto_data) == 1:
    axes = [axes]

for idx, (ticker, prices) in enumerate(crypto_data.items()):
    if len(prices) == 0:
        continue
    daily = prices.resample('D').last().dropna()
    roc_20 = ((daily / daily.shift(20)) - 1) * 100
    
    ax = axes[idx]
    ax.plot(roc_20.index, roc_20.values, color=colors.get(ticker, 'gray'), linewidth=1.5)
    ax.fill_between(roc_20.index, roc_20.values, 0, 
                    where=(roc_20.values >= 0), alpha=0.3, color='green')
    ax.fill_between(roc_20.index, roc_20.values, 0, 
                    where=(roc_20.values < 0), alpha=0.3, color='red')
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axhline(y=20, color='green', linestyle='--', alpha=0.5, label='Strong (+20%)')
    ax.axhline(y=-20, color='red', linestyle='--', alpha=0.5, label='Weak (-20%)')
    ax.set_title(f'{ticker} 20-Day Momentum (Rate of Change)', fontweight='bold')
    ax.set_ylabel('ROC (%)')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.xlabel('Date', fontsize=12)
plt.suptitle('Momentum Signals: 20-Day Rate of Change', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Green = positive momentum, Red = negative momentum.')

In [ ]:
# Mean Reversion: Z-Score of Daily Returns
# Z-score = (return - mean) / std — measures how unusual a day's return is

fig, axes = plt.subplots(len(crypto_data), 1, figsize=(12, 3 * len(crypto_data)), sharex=True)
if len(crypto_data) == 1:
    axes = [axes]

zscore_data = {}

for idx, (ticker, ret) in enumerate(returns_data.items()):
    z = (ret - ret.mean()) / ret.std()
    zscore_data[ticker] = z
    
    ax = axes[idx]
    ax.plot(z.index, z.values, color=colors.get(ticker, 'gray'), linewidth=1, alpha=0.7)
    ax.axhline(y=0, color='black', linewidth=0.5)
    ax.axhline(y=2, color='red', linestyle='--', linewidth=1, label='Overbought (+2)')
    ax.axhline(y=-2, color='green', linestyle='--', linewidth=1, label='Oversold (-2)')
    
    # Highlight extreme signals
    ax.fill_between(z.index, z.values, 2, where=(z.values >= 2), 
                    alpha=0.4, color='red', label='Sell Signal')
    ax.fill_between(z.index, z.values, -2, where=(z.values <= -2), 
                    alpha=0.4, color='green', label='Buy Signal')
    
    ax.set_title(f'{ticker} Return Z-Score (Mean Reversion)', fontweight='bold')
    ax.set_ylabel('Z-Score')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.xlabel('Date', fontsize=12)
plt.suptitle('Mean Reversion Analysis: Z-Score of Daily Returns', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Z-score > +2 suggests overbought (mean reversion down likely).')
print('Z-score < -2 suggests oversold (mean reversion up likely).')

In [ ]:
# Simple Backtest: Mean Reversion Strategy
# Buy when z-score < -2, sell when z-score > +2

print('MEAN REVERSION BACKTEST')
print('=' * 60)

backtest_results = []

for ticker in returns_data.keys():
    ret = returns_data[ticker]
    z = zscore_data[ticker]
    
    # Strategy: buy when z < -2 (oversold), exit when z > +2 (overbought)
    # Position: 1 when z < -2, -1 when z > +2, 0 otherwise
    position = pd.Series(0, index=ret.index)
    position[z < -2] = 1   # Long (oversold)
    position[z > 2] = -1   # Short (overbought)
    
    # Shift position by 1 day to avoid lookahead bias
    position = position.shift(1).fillna(0)
    
    # Strategy returns
    strat_ret = position * ret
    
    # Buy and hold returns
    bh_ret = ret.copy()
    
    # Cumulative returns
    strat_cum = (1 + strat_ret / 100).cumprod()
    bh_cum = (1 + bh_ret / 100).cumprod()
    
    backtest_results.append({
        'Asset': ticker,
        'Strategy Return': strat_cum.iloc[-1] - 1 if len(strat_cum) > 0 else 0,
        'BuyHold Return': bh_cum.iloc[-1] - 1 if len(bh_cum) > 0 else 0,
        'Outperformance': (strat_cum.iloc[-1] - bh_cum.iloc[-1]) if len(strat_cum) > 0 and len(bh_cum) > 0 else 0,
        'Days in Market': (position != 0).sum(),
    })

bt_df = pd.DataFrame(backtest_results)
display(bt_df.round(4))
print()
print('Strategy: Long when z-score < -2, Short when z-score > +2')
print('All calculations on REAL CoinGecko data.')

In [ ]:
# Plot Strategy vs Buy-and-Hold Performance
fig, axes = plt.subplots(1, len(returns_data), figsize=(6 * len(returns_data), 5))
if len(returns_data) == 1:
    axes = [axes]

for idx, ticker in enumerate(returns_data.keys()):
    ret = returns_data[ticker]
    z = zscore_data[ticker]
    
    position = pd.Series(0, index=ret.index)
    position[z < -2] = 1
    position[z > 2] = -1
    position = position.shift(1).fillna(0)
    
    strat_ret = position * ret
    strat_cum = (1 + strat_ret / 100).cumprod()
    bh_cum = (1 + ret / 100).cumprod()
    
    ax = axes[idx]
    ax.plot(bh_cum.index, bh_cum.values, label='Buy & Hold', 
            color=colors.get(ticker, 'gray'), linewidth=2, alpha=0.7)
    ax.plot(strat_cum.index, strat_cum.values, label='Mean Reversion', 
            color='red', linewidth=1.5)
    ax.set_title(f'{ticker}: Strategy vs Buy & Hold', fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Cumulative Return')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Backtest: Mean Reversion vs Buy & Hold', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Note: This is a simplified backtest for demonstration. Real trading includes fees, slippage, etc.')

In [ ]:
# Final Summary
print('=' * 70)
print('CRYPTO ANALYSIS SUMMARY')
print('=' * 70)

print('\n1. DATA SOURCES')
print('   All price data fetched live from CoinGecko API (free tier)')
print(f'   Time period: Last 365 days')

print('\n2. ASSET PERFORMANCE')
for ticker, prices in crypto_data.items():
    if len(prices) == 0:
        continue
    ret = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
    ann_vol = returns_data[ticker].std() * np.sqrt(365) if ticker in returns_data else 0
    print(f'   {ticker}: ${prices.iloc[0]:,.0f} -> ${prices.iloc[-1]:,.0f} ({ret:+.1f}%) | Vol: {ann_vol:.1f}%')

print('\n3. CORRELATIONS (Daily Returns)')
for i, row in corr.iterrows():
    for j, val in row.items():
        if i < j:
            print(f'   {i}-{j}: {val:.3f}')

print('\n4. KEY INSIGHTS')
print('   - Crypto assets show high correlation during stress periods')
print('   - Volatility is significantly higher than traditional assets')
print('   - Mean reversion signals (z-score) can identify extremes')
print('   - Past performance does not guarantee future results')

print('\n5. RATE LIMIT NOTES')
print('   CoinGecko free tier: 10-30 calls/minute')
print('   If rate-limited, wait 60 seconds and re-run the fetch cell.')

print('=' * 70)
print('All data is REAL from CoinGecko. No synthetic data used.')
print('=' * 70)